In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pyarrow as pa
import pyarrow.compute as pc
from typing import Optional, List


# srun -w mauao -c 5 --partition=interactive --pty bash
# srun -w ngongotaha -c 5 --partition=interactive --pty bash
# poetry shell
# jupyter server --no-browser --port=8889
# ssh -L 8889:localhost:8889 ls985@mauao
# ssh -L 8889:localhost:8889 ls985@ngongotaha

In [2]:
RESULTS_DIR = Path("/nfs-share/ls985/pollen_worker/results")

In [39]:
def read_logs_flwr(log_file_path: Path) -> Optional[pa.Table]:
    """Reads the log file and returns a pandas dataframe with the data."""
    # Init data for the table
    hardware_setting = 0 # 1 or 2
    n_clients_per_round = 0
    round_numbers: List[int] = []
    start_timestamp: List[pd.Timestamp] = []
    end_timestamp: List[pd.Timestamp] = []
    previous_timestamp: Optional[pd.Timestamp] = None
    # Opening log file
    with open(log_file_path, 'r') as f:
        lines = f.readlines()
    # Reading first configuration line
    line = lines.pop(0)
    strings = line.split(' ')
    # Reading first timestamp
    date = strings[0].replace('[', '')
    time = strings[1].split(']')[0]
    previous_timestamp = pd.to_datetime(f"{date} {time}", format="%Y-%m-%d %H:%M:%S,%f")
    # Reading dataset
    dataset = strings[5]
    # Reading framework
    framework = 'Flower+Ray' if 'ray_simulation' in log_file_path.name else f'Pollen-{strings[13].split("=")[1].upper()}'
    # Reading log file
    cnt = 0
    for line in lines:
        if line.startswith('['):
            strings = line.split(' ')
            if hardware_setting == 0:
                if framework == 'Flower+Ray' and "\'GPU\':" in strings:
                    idx = strings.index("\'GPU\':")
                    hardware_setting = int(strings[idx+1].split('.')[0])
                elif framework == 'Pollen' and ("sending" in strings and "instructions" in strings and "NodeManagers" in strings):
                    idx = strings.index("NodeManagers")
                    hardware_setting = int(strings[idx-1])
            
            if ('fit_round' in strings) and ('strategy' in strings):
                date = strings[0].replace('[', '')
                time = strings[1].split(']')[0]
                last_timestamp = pd.to_datetime(f"{date} {time}", format="%Y-%m-%d %H:%M:%S,%f")
                if n_clients_per_round == 0:
                    n_clients_per_round = int(strings[7])
                round_numbers.append(cnt)
                cnt+=1
                start_timestamp.append(previous_timestamp)
                previous_timestamp = last_timestamp
                end_timestamp.append(last_timestamp)
    # Creating dict to pass to pa.Table builder
    dict_for_table = {
        'rct': [(t1 - t0) for t0, t1 in zip(start_timestamp, end_timestamp)],
        'round': round_numbers,
        'n_clients_per_round': [n_clients_per_round]*len(round_numbers),
        'dataset': [dataset]*len(round_numbers),
        'framework': [framework]*len(round_numbers),
        'hardware_setting': [hardware_setting]*len(round_numbers),
        'is_dropped': [False]*len(round_numbers),
    }
    # Returning pa.Table
    return pa.Table.from_pydict(dict_for_table)
                

In [40]:
def read_logs_fedscale(log_file_path: Path) -> Optional[pa.Table]:
    """Reads the log file and returns a pandas dataframe with the data."""
    # Init data for the table
    hardware_setting = 0 # 1 or 2
    n_clients_per_round = 0
    round_numbers: List[int] = []
    start_timestamp: List[pd.Timestamp] = []
    end_timestamp: List[pd.Timestamp] = []
    is_dropped: List[bool] = []
    previous_timestamp: Optional[pd.Timestamp] = None
    # Opening log file
    with open(log_file_path, 'r') as f:
        lines = f.readlines()
    # Reading first configuration line
    line = lines.pop(0)
    strings = line.split(' ')
    while '' in strings:
        strings.remove('')
    # Reading first timestamp
    date = strings[0] # format is "(mm-dd)"
    time = strings[1] # format is "hh:mm:ss"
    previous_timestamp = pd.to_datetime(f"(2023-{date}) {time}", format="(%Y-(%m-%d)) %H:%M:%S")
    # Reading dataset
    dataset = strings[6].split("\'")[1].split("-")[0]
    # Reading framework
    framework = 'FedScale'
    # Reading the hardware setting
    hardware_setting = len(strings[16].split("\'")[1].split("="))
    # Reading log file
    cnt = 0
    for line in lines:
        to_be_discarded = False
        strings = line.split(' ')
        while '' in strings:
            strings.remove('')
        if 'morons.' in strings:
            to_be_discarded = True
        if ('Empty'  in strings) and ('results'  in strings) and ('from'  in strings) and ('client' in strings):
            to_be_discarded = True
        if '[aggregator.py:543]' in strings and 'round:' in strings:
            date = strings[0] # format is "(mm-dd)"
            time = strings[1] # format is "hh:mm:ss"
            last_timestamp = pd.to_datetime(f"(2023-{date}) {time}", format="(%Y-(%m-%d)) %H:%M:%S")
            if to_be_discarded:
                print(f"Round {cnt} to be discarded due to morons or failed deserilization")
            if n_clients_per_round == 0:
                n_clients_per_round = int(int(strings[12].replace(',', ''))*(1.0/1.3))
            to_be_discarded = False
            round_numbers.append(cnt)
            cnt+=1
            start_timestamp.append(previous_timestamp)
            previous_timestamp = last_timestamp
            end_timestamp.append(last_timestamp)
            is_dropped.append(to_be_discarded)
    # Creating dict to pass to pa.Table builder
    dict_for_table = {
        'rct': [(t1 - t0) for t0, t1 in zip(start_timestamp, end_timestamp)],
        'round': round_numbers,
        'n_clients_per_round': [n_clients_per_round]*len(round_numbers),
        'dataset': [dataset]*len(round_numbers),
        'framework': [framework]*len(round_numbers),
        'hardware_setting': [hardware_setting]*len(round_numbers),
        'is_dropped': is_dropped,
    }
    # Returning pa.Table
    return pa.Table.from_pydict(dict_for_table)

In [71]:
def read_logs_flute(log_file_path: Path):
    """Reads the log file and returns a pandas dataframe with the data."""
    # Init data for the table
    n_clients_per_round = 0
    round_numbers: List[int] = []
    start_timestamp: List[pd.Timestamp] = []
    end_timestamp: List[pd.Timestamp] = []
    previous_timestamp: Optional[pd.Timestamp] = None
    # Opening log file
    with open(log_file_path, 'r') as f:
        lines = f.readlines()
    # Reading the hardware setting
    line = lines.pop(3)
    hardware_setting = 1 if int(line[-2]) == 1 else 2
    # Reading dataset
    line = lines.pop(5)
    strings = line.split(' ')
    dataset = strings[5].replace("\'", "").replace(",", "")
    # Reading framework
    framework = 'Flute'
    # Reading the first timestamp
    while previous_timestamp is None:
        line = lines.pop(0)
        strings = line.split(' ')
        while '' in strings:
            strings.remove('')
        if ('Assigning' in strings and 'default' in strings and 'values' in strings):
            datestring = line.split(' : ')[0].replace(' ', '-')
            previous_timestamp = pd.to_datetime(datestring, format="%a-%b-%d-%H:%M:%S-%Y")
    # Reading log file
    cnt = 0
    for line in lines:
        strings = line.split(' ')
        if n_clients_per_round == 0:
            if ('Clients' in strings and 'for' in strings and 'round' in strings):
                n_clients_per_round = int(strings[9])
        if 'iteration' in strings:
            datestring = line.split(' : ')[0].replace(' ', '-')
            last_timestamp = pd.to_datetime(datestring, format="%a-%b-%d-%H:%M:%S-%Y")
            round_numbers.append(cnt)
            cnt+=1
            start_timestamp.append(previous_timestamp)
            previous_timestamp = last_timestamp
            end_timestamp.append(last_timestamp)
    # Creating dict to pass to pa.Table builder
    dict_for_table = {
        'rct': [(t1 - t0) for t0, t1 in zip(start_timestamp, end_timestamp)],
        'round': round_numbers,
        'n_clients_per_round': [n_clients_per_round]*len(round_numbers),
        'dataset': [dataset]*len(round_numbers),
        'framework': [framework]*len(round_numbers),
        'hardware_setting': [hardware_setting]*len(round_numbers),
        'is_dropped': [False]*len(round_numbers),
    }
    # Returning pa.Table
    return pa.Table.from_pydict(dict_for_table)

In [74]:
results_table: Optional[pa.Table] = None
for result_file in RESULTS_DIR.glob('*'):
    tmp_table: Optional[pa.Table] = None
    filename = result_file.name
    if filename.startswith('log'):
        tmp_table = read_logs_fedscale(result_file)
    elif filename.startswith('flute'):
        tmp_table = read_logs_flute(result_file)
    else:
        tmp_table = read_logs_flwr(result_file)
    if tmp_table is not None:
        if results_table is None:
            results_table = tmp_table
        else:
            results_table = pa.concat_tables(
                [results_table, tmp_table],
            )

In [75]:
results_table

pyarrow.Table
rct: duration[us]
round: int64
n_clients_per_round: int64
dataset: string
framework: string
hardware_setting: int64
is_dropped: bool
----
rct: [[73000000],[11000000,39000000,24000000,24000000,26000000,...,27000000,26000000,20000000,29000000,21000000],...,[6365000,3343204000],[35000000,127000000,105000000,102000000,102000000,...,99000000,101000000,100000000,97000000,98000000]]
round: [[0],[0,1,2,3,4,...,95,96,97,98,99],...,[0,1],[0,1,2,3,4,...,105,106,107,108,109]]
n_clients_per_round: [[0],[100,100,100,100,100,...,100,100,100,100,100],...,[10000,10000],[99,99,99,99,99,...,99,99,99,99,99]]
dataset: [["shakespeare"],["here:","here:","here:","here:","here:",...,"here:","here:","here:","here:","here:"],...,["reddit","reddit"],["google_speech","google_speech","google_speech","google_speech","google_speech",...,"google_speech","google_speech","google_speech","google_speech","google_speech"]]
framework: [["FedScale"],["Flute","Flute","Flute","Flute","Flute",...,"Flute","Flute","

In [73]:
results_table.filter(pc.field('framework') == pc.scalar('Flute'))

pyarrow.Table
rct: duration[us]
round: int64
n_clients_per_round: int64
dataset: string
framework: string
hardware_setting: int64
is_dropped: bool
----
rct: [[11000000,39000000,24000000,24000000,26000000,...,27000000,26000000,20000000,29000000,21000000]]
round: [[0,1,2,3,4,...,95,96,97,98,99]]
n_clients_per_round: [[100,100,100,100,100,...,100,100,100,100,100]]
dataset: [["here:","here:","here:","here:","here:",...,"here:","here:","here:","here:","here:"]]
framework: [["Flute","Flute","Flute","Flute","Flute",...,"Flute","Flute","Flute","Flute","Flute"]]
hardware_setting: [[1,1,1,1,1,...,1,1,1,1,1]]
is_dropped: [[false,false,false,false,false,...,false,false,false,false,false]]